[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/certified-journeys.github.io/blob/main/courses/claude-api-certified/notebooks/day-08-mcp.ipynb#scrollTo=ea01eb02)

---
# Day 8 · Model Context Protocol
**certified-journeys / claude-api-certified** · MCP servers, clients & resources

> **Goal for today:** Understand what MCP is, build a working MCP server with a custom tool and resource, then connect to it with an MCP client — the same pattern used by Claude Desktop and Claude Code.

In [ ]:
%pip install -q anthropic mcp

In [ ]:
import os
import json
import asyncio
import anthropic

try:
    from google.colab import userdata
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
except ImportError:
    pass

client = anthropic.Anthropic()
print("anthropic + mcp packages loaded.")

## Step 1 · What is MCP and why it matters

The Model Context Protocol is an open standard for connecting AI models to external data and tools. It separates **tool definition** from **tool execution** — you build a server once and reuse it across any MCP-compatible client.

```
  MCP Client            MCP Server
  (Claude Desktop,  ←→  (your code)
   Claude Code,
   your app)             ↓
                      Tools · Resources · Prompts
```

| Component | Purpose |
|---|---|
| **Tool** | A function Claude can call (like the Day 5 tools, but over a protocol) |
| **Resource** | Read-only data Claude can access (files, APIs, databases) |
| **Prompt** | Pre-built prompt templates stored server-side |
| **Transport** | How client/server communicate (stdio, HTTP) |

In [ ]:
# MCP server code — write to a file, then run it as a subprocess
SERVER_CODE = '''
from mcp.server.fastmcp import FastMCP
import json
import datetime

mcp = FastMCP("certified-journeys-demo")

@mcp.tool()
def get_time(timezone: str = "UTC") -> str:
    """Get the current time. Optionally specify a timezone name."""
    now = datetime.datetime.utcnow()
    return json.dumps({"time": now.strftime("%H:%M:%S"), "timezone": timezone, "date": now.strftime("%Y-%m-%d")})


@mcp.tool()
def word_count(text: str) -> str:
    """Count words, characters, and sentences in a text string."""
    words = len(text.split())
    chars = len(text)
    sentences = text.count(".") + text.count("!") + text.count("?")
    return json.dumps({"words": words, "characters": chars, "sentences": sentences})


@mcp.resource("config://app-settings")
def get_app_settings() -> str:
    """Return application configuration as JSON."""
    return json.dumps({
        "name": "certified-journeys",
        "version": "1.0.0",
        "courses": ["claude-api-certified", "dlt-certified", "polars-certified"],
        "features": ["progress-tracking", "github-sync", "notebooks"]
    }, indent=2)


if __name__ == "__main__":
    mcp.run(transport="stdio")
'''

# Write the server to a file
with open("mcp_server.py", "w") as f:
    f.write(SERVER_CODE)

print("MCP server written to mcp_server.py")
print("\nServer exposes:")
print("  Tools: get_time, word_count")
print("  Resources: config://app-settings")

**What just happened?**
- `FastMCP` is a high-level Python SDK for building MCP servers with decorators.
- **`@mcp.tool()`** registers a function as a callable tool — the docstring becomes the tool description.
- **`@mcp.resource("config://app-settings")`** registers a resource at a URI — clients can read it.
- `mcp.run(transport="stdio")` starts the server over stdin/stdout — the simplest transport for local use.

## Step 2 · Connect with an MCP client

An MCP client connects to a server, discovers available tools/resources, and exposes them to Claude.

In [ ]:
import subprocess
import sys
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client


async def list_server_capabilities():
    """Connect to the MCP server and list all available tools and resources."""
    server_params = StdioServerParameters(
        command=sys.executable,
        args=["mcp_server.py"]
    )

    async with stdio_client(server_params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()

            # List tools
            tools = await session.list_tools()
            print("Available tools:")
            for tool in tools.tools:
                print(f"  - {tool.name}: {tool.description}")

            # List resources
            resources = await session.list_resources()
            print("\nAvailable resources:")
            for resource in resources.resources:
                print(f"  - {resource.uri}: {resource.description}")


await list_server_capabilities()

**What just happened?**
- `stdio_client()` launches `mcp_server.py` as a subprocess and communicates via stdin/stdout.
- `session.initialize()` performs the MCP handshake — required before any other calls.
- `list_tools()` and `list_resources()` return what the server currently exposes — dynamic discovery.
- **This is how Claude Desktop works** — it launches your MCP server as a subprocess and talks to it over stdio.

## Step 3 · Call MCP tools from the client

Once connected, you can call any tool the server exposes and read any resource.

In [ ]:
async def use_mcp_tools():
    """Call tools and read resources from the MCP server."""
    server_params = StdioServerParameters(
        command=sys.executable,
        args=["mcp_server.py"]
    )

    async with stdio_client(server_params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()

            # Call get_time tool
            time_result = await session.call_tool("get_time", {"timezone": "UTC"})
            print("get_time result:", time_result.content[0].text)

            # Call word_count tool
            sample_text = "The Model Context Protocol connects AI models to external tools and data sources."
            count_result = await session.call_tool("word_count", {"text": sample_text})
            print("word_count result:", count_result.content[0].text)

            # Read a resource
            resource_content, mime = await session.read_resource("config://app-settings")
            print("\nResource content:")
            print(resource_content[:200])


await use_mcp_tools()

**What just happened?**
- `session.call_tool(name, arguments)` invokes a tool and returns a `CallToolResult`.
- `session.read_resource(uri)` reads a resource and returns `(content_string, mime_type)`.
- **Tool results** are wrapped in content blocks — same pattern as Claude API tool_result blocks.
- The client doesn't need to know the tool implementation — it just knows the name and schema.

## Step 4 · Connect MCP tools to Claude

The real power: use the MCP client to discover tools, convert them to Claude tool schemas, and let Claude decide when to call them.

In [ ]:
async def claude_with_mcp(user_question: str) -> str:
    """Use MCP tools with Claude in a tool loop."""
    server_params = StdioServerParameters(
        command=sys.executable,
        args=["mcp_server.py"]
    )

    async with stdio_client(server_params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()

            # Convert MCP tools to Claude tool schemas
            mcp_tools = await session.list_tools()
            claude_tools = [
                {
                    "name": t.name,
                    "description": t.description or "",
                    "input_schema": t.inputSchema
                }
                for t in mcp_tools.tools
            ]

            messages = [{"role": "user", "content": user_question}]

            # Tool loop
            for _ in range(5):  # max 5 iterations
                r = client.messages.create(
                    model="claude-haiku-4-5",
                    max_tokens=512,
                    tools=claude_tools,
                    messages=messages
                )
                messages.append({"role": "assistant", "content": r.content})

                if r.stop_reason == "end_turn":
                    return next((b.text for b in r.content if b.type == "text"), "")

                if r.stop_reason == "tool_use":
                    tool_results = []
                    for block in r.content:
                        if block.type == "tool_use":
                            # Call via MCP client (not direct function)
                            mcp_result = await session.call_tool(block.name, block.input)
                            result_text = mcp_result.content[0].text if mcp_result.content else "{}"
                            tool_results.append({
                                "type": "tool_result",
                                "tool_use_id": block.id,
                                "content": result_text
                            })
                    messages.append({"role": "user", "content": tool_results})

    return "No response generated."


answer = await claude_with_mcp("What time is it and how many words are in 'Hello world, this is MCP'?")
print("Claude's answer:", answer)

**What just happened?**
- MCP tools are **dynamically discovered** at runtime — Claude sees whatever the server currently exposes.
- **Tool execution goes through the MCP client**, not direct function calls — the server could be remote.
- This is the pattern Claude Desktop uses: Claude talks to the API, the API calls your app, your app calls the MCP server.
- Swapping the server (e.g., pointing to a different MCP server) requires zero changes to the Claude integration code.

## Step 5 · Add error handling and logging

Production MCP integrations need structured error handling for tool failures and connection issues.

In [ ]:
import logging

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger(__name__)


async def safe_mcp_call(session, tool_name: str, tool_input: dict) -> str:
    """Call an MCP tool with error handling and structured logging."""
    logger.info(f"Calling MCP tool: {tool_name}({json.dumps(tool_input)})")
    try:
        result = await session.call_tool(tool_name, tool_input)
        text = result.content[0].text if result.content else "{}"
        logger.info(f"MCP tool {tool_name} succeeded: {text[:80]}")
        return text
    except Exception as e:
        logger.error(f"MCP tool {tool_name} failed: {e}")
        return json.dumps({"error": str(e), "tool": tool_name})


# Test safe calling
async def test_safe_call():
    server_params = StdioServerParameters(command=sys.executable, args=["mcp_server.py"])
    async with stdio_client(server_params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            # Valid call
            result = await safe_mcp_call(session, "word_count", {"text": "test input"})
            print("Valid call result:", result)
            # Invalid tool (will error)
            result = await safe_mcp_call(session, "nonexistent_tool", {})
            print("Invalid call result:", result)


await test_safe_call()

**What just happened?**
- **Structured logging** captures every tool call — essential for debugging production MCP integrations.
- Error handling returns a JSON error object instead of raising — Claude can read the error and adjust.
- **Log tool inputs and outputs** (with truncation for large payloads) so you can reproduce failures.
- Consider adding metrics: call duration, success rate, and error types per tool.

In [ ]:
# Challenge: Add a search_courses tool to the MCP server
# The tool should:
# - Accept a query string
# - Search through a hardcoded course list
# - Return matching courses as JSON
# Modify mcp_server.py to add the tool, then test it with the client

NEW_TOOL_CODE = '''
@mcp.tool()
def search_courses(query: str) -> str:
    """Search the certified-journeys course catalog by keyword."""
    # Your solution here
    # courses = [...] — define the catalog
    # Filter by query.lower() in course name or tags
    # Return JSON list of matching courses
    pass
'''

print("Add this tool to mcp_server.py:")
print(NEW_TOOL_CODE)
print("Then test: await claude_with_mcp('Which courses cover Python?')")

---
## Day 8 key concepts recap

| Concept | What to remember |
|---|---|
| MCP Tool | `@mcp.tool()` decorator — docstring = Claude's description |
| MCP Resource | `@mcp.resource(uri)` — read-only data at a URI |
| stdio transport | Server runs as subprocess; client communicates via stdin/stdout |
| `session.initialize()` | Required before any other MCP call |
| Claude + MCP | Discover tools, convert to Claude schemas, run tool loop |

> **Tip:** MCP is the USB-C of AI tool integrations. Build a server once and reuse it across Claude Desktop, Claude Code, and your own apps.

---
## What's next
**Day 9** → Agents & Claude Code: ReAct loops, error budgets, Computer Use, and multi-agent patterns.

Mark Day 8 complete in your [tracker](../index.html).